# Week 4 Day 1 — SQL for ML

## What I built:
- SQLite database from Titanic CSV using pandas to_sql()
- 5 SQL queries — survival by gender, fare by class,
  oldest passengers, null counts, embarkation analysis
- Interview question query — survival by gender + class
- Verified SQL results match pandas groupby exactly
- Demonstrated INNER JOIN vs LEFT JOIN with small example

## Key SQL concepts:

**SELECT, WHERE, GROUP BY, ORDER BY, LIMIT:**
- WHERE filters rows before grouping
- HAVING filters groups after grouping
- ORDER BY DESC → highest first
- LIMIT → top N rows only

**Aggregate functions:**
- COUNT(*) → total rows
- SUM(col) → total of column
- AVG(col) → average
- MIN/MAX  → smallest/largest
- ROUND(val, 2) → 2 decimal places

**JOINs:**
- INNER JOIN → only matching rows in both tables
- LEFT JOIN  → all left rows + matching right (NULL if no match)

**to_sql() parameters:**
- conn        → which database to save to
- index=False → don't save pandas index as SQL column
- if_exists='replace' → delete old table, create fresh

## Key finding:
Female 1st class → 96.81% survival (highest)
Male 3rd class   → 13.54% survival (lowest)
Gap = 83.27% — most powerful Titanic insight in one SQL query
SQL and Pandas give identical results ✅

# Set up SQLite with Titanic:

In [1]:
import sqlite3
import pandas as pd
import numpy as np

# load Titanic
df = pd.read_csv('train.csv')

# create SQLite database in memory
conn = sqlite3.connect(':memory:')

# load DataFrame into SQLite table
df.to_sql('titanic', conn, index=False, if_exists='replace')

print("Database created successfully")
print(f"Rows loaded: {pd.read_sql('SELECT COUNT(*) as total FROM titanic', conn).iloc[0,0]}")

Database created successfully
Rows loaded: 891


# Run your first 5 SQL queries:

In [2]:
# Query 1 — survival count by gender
q1 = pd.read_sql("""
    SELECT Sex, 
           COUNT(*) as total,
           SUM(Survived) as survived,
           ROUND(AVG(Survived) * 100, 2) as survival_rate_pct
    FROM titanic
    GROUP BY Sex
    ORDER BY survival_rate_pct DESC
""", conn)
print("Query 1 — Survival by Gender:")
print(q1)

# Query 2 — avg fare by class
q2 = pd.read_sql("""
    SELECT Pclass,
           COUNT(*) as passengers,
           ROUND(AVG(Fare), 2) as avg_fare,
           ROUND(MIN(Fare), 2) as min_fare,
           ROUND(MAX(Fare), 2) as max_fare
    FROM titanic
    GROUP BY Pclass
    ORDER BY Pclass
""", conn)
print("\nQuery 2 — Fare by Class:")
print(q2)

# Query 3 — top 10 oldest passengers
q3 = pd.read_sql("""
    SELECT Name, Age, Sex, Pclass, Survived
    FROM titanic
    WHERE Age IS NOT NULL
    ORDER BY Age DESC
    LIMIT 10
""", conn)
print("\nQuery 3 — Top 10 Oldest Passengers:")
print(q3)

# Query 4 — null count per column
q4 = pd.read_sql("""
    SELECT 
        SUM(CASE WHEN Age IS NULL THEN 1 ELSE 0 END) as Age_nulls,
        SUM(CASE WHEN Cabin IS NULL THEN 1 ELSE 0 END) as Cabin_nulls,
        SUM(CASE WHEN Embarked IS NULL THEN 1 ELSE 0 END) as Embarked_nulls
    FROM titanic
""", conn)
print("\nQuery 4 — Null Counts:")
print(q4)

# Query 5 — passenger count per embarkation port
q5 = pd.read_sql("""
    SELECT Embarked,
           COUNT(*) as total,
           ROUND(AVG(Survived) * 100, 2) as survival_rate_pct
    FROM titanic
    WHERE Embarked IS NOT NULL
    GROUP BY Embarked
    ORDER BY total DESC
""", conn)
print("\nQuery 5 — Embarkation Port Analysis:")
print(q5)

Query 1 — Survival by Gender:
      Sex  total  survived  survival_rate_pct
0  female    314       233              74.20
1    male    577       109              18.89

Query 2 — Fare by Class:
   Pclass  passengers  avg_fare  min_fare  max_fare
0       1         216     84.15       0.0    512.33
1       2         184     20.66       0.0     73.50
2       3         491     13.68       0.0     69.55

Query 3 — Top 10 Oldest Passengers:
                                   Name   Age   Sex  Pclass  Survived
0  Barkworth, Mr. Algernon Henry Wilson  80.0  male       1         1
1                   Svensson, Mr. Johan  74.0  male       3         0
2             Goldschmidt, Mr. George B  71.0  male       1         0
3               Artagaveytia, Mr. Ramon  71.0  male       1         0
4                  Connors, Mr. Patrick  70.5  male       3         0
5           Mitchell, Mr. Henry Michael  70.0  male       2         0
6          Crosby, Capt. Edward Gifford  70.0  male       1         0
7

# The interview question query:

In [3]:
# Classic interview question:
# "Find survival rate for each gender-class combination,
#  ordered by survival rate descending"

q6 = pd.read_sql("""
    SELECT Sex,
           Pclass,
           COUNT(*) as total,
           SUM(Survived) as survived,
           ROUND(AVG(Survived) * 100, 2) as survival_rate_pct
    FROM titanic
    GROUP BY Sex, Pclass
    ORDER BY survival_rate_pct DESC
""", conn)
print("Survival rate by Gender + Class:")
print(q6)

# verify with pandas groupby — should be identical
pandas_result = df.groupby(['Sex', 'Pclass'])['Survived'].agg(
    total='count',
    survived='sum',
    survival_rate_pct=lambda x: round(x.mean() * 100, 2)
).reset_index()
print("\nPandas verification:")
print(pandas_result.sort_values('survival_rate_pct', ascending=False))

Survival rate by Gender + Class:
      Sex  Pclass  total  survived  survival_rate_pct
0  female       1     94        91              96.81
1  female       2     76        70              92.11
2  female       3    144        72              50.00
3    male       1    122        45              36.89
4    male       2    108        17              15.74
5    male       3    347        47              13.54

Pandas verification:
      Sex  Pclass  total  survived  survival_rate_pct
0  female       1     94        91              96.81
1  female       2     76        70              92.11
2  female       3    144        72              50.00
3    male       1    122        45              36.89
4    male       2    108        17              15.74
5    male       3    347        47              13.54


# Create two tables and practice JOINs:

In [4]:
# create passengers table
passengers = df[['PassengerId', 'Name', 'Age', 'Sex']].copy()
passengers.to_sql('passengers', conn, index=False, if_exists='replace')

# create tickets table
tickets = df[['PassengerId', 'Pclass', 'Fare', 'Survived']].copy()
tickets.to_sql('tickets', conn, index=False, if_exists='replace')

# INNER JOIN
inner = pd.read_sql("""
    SELECT p.PassengerId, p.Name, p.Sex, p.Age,
           t.Pclass, t.Fare, t.Survived
    FROM passengers p
    INNER JOIN tickets t ON p.PassengerId = t.PassengerId
    LIMIT 5
""", conn)
print("INNER JOIN result:")
print(inner)

# LEFT JOIN — same result here since all IDs match
# demonstrate difference with missing IDs
pd.DataFrame({'PassengerId': [1,2,3], 
              'Name': ['A','B','C']}).to_sql('p_small', conn, 
                                              index=False, 
                                              if_exists='replace')
pd.DataFrame({'PassengerId': [1,2], 
              'Fare': [100,200]}).to_sql('t_small', conn, 
                                          index=False, 
                                          if_exists='replace')

inner_small = pd.read_sql("""
    SELECT p.PassengerId, p.Name, t.Fare
    FROM p_small p
    INNER JOIN t_small t ON p.PassengerId = t.PassengerId
""", conn)

left_small = pd.read_sql("""
    SELECT p.PassengerId, p.Name, t.Fare
    FROM p_small p
    LEFT JOIN t_small t ON p.PassengerId = t.PassengerId
""", conn)

print("\nINNER JOIN (small):")
print(inner_small)
print("\nLEFT JOIN (small):")
print(left_small)
print("\nDifference — LEFT JOIN keeps passenger C with NULL fare")

INNER JOIN result:
   PassengerId                                               Name     Sex  \
0            1                            Braund, Mr. Owen Harris    male   
1            2  Cumings, Mrs. John Bradley (Florence Briggs Th...  female   
2            3                             Heikkinen, Miss. Laina  female   
3            4       Futrelle, Mrs. Jacques Heath (Lily May Peel)  female   
4            5                           Allen, Mr. William Henry    male   

    Age  Pclass     Fare  Survived  
0  22.0       3   7.2500         0  
1  38.0       1  71.2833         1  
2  26.0       3   7.9250         1  
3  35.0       1  53.1000         1  
4  35.0       3   8.0500         0  

INNER JOIN (small):
   PassengerId Name  Fare
0            1    A   100
1            2    B   200

LEFT JOIN (small):
   PassengerId Name   Fare
0            1    A  100.0
1            2    B  200.0
2            3    C    NaN

Difference — LEFT JOIN keeps passenger C with NULL fare
